# Session 14: MCP-Powered X Post Summarizer

## Building a LangGraph Agent with GitHub MCP Tools, X API Tools, and Memory

## Learning Objectives:

- **Ingest MCP servers as LangGraph tools** using `langchain-mcp-adapters` to connect to the GitHub MCP Server and use its tools programmatically
- **Wrap the X (Twitter) API as a LangChain tool** using the `@tool` decorator so a LangGraph agent can search and retrieve public posts
- **Build a LangGraph agent with memory** that combines MCP-sourced tools and custom tools, using `MemorySaver` for short-term conversational memory
- **Orchestrate a full workflow through the agent** — search X posts, generate summaries, create a GitHub repo, commit files, branch, and open a PR — all via natural language

## Overview

In this notebook, you will build a **LangGraph ReAct agent** that has access to two categories of tools:

1. **GitHub MCP tools** — loaded from the official GitHub MCP Server via `langchain-mcp-adapters`. These replace manual `git` commands with tool calls the agent can invoke (create repos, commit files, create branches, open PRs).
2. **X API tools** — custom Python functions wrapped with the `@tool` decorator that call the X API v2 directly to search and retrieve posts.

The agent uses **`MemorySaver`** for short-term memory so it can maintain context across multi-step workflows within a conversation thread.

There will be one breakout room with two phases:

- 🤝 Phase 1: Setup, Tools & Agent Construction
  - Task 1: Dependencies & Environment
  - Task 2: X API as LangChain Tools
  - Task 3: Connect to GitHub MCP Server & Load Tools
  - Task 4: Build the LangGraph Agent with Memory
  - Task 5: Test the Agent — Search & Summarize X Posts
  - Activity #1: Extend the Agent with a Custom X API Tool
- 🤝 Phase 2: MCP Workflow Through the Agent
  - Task 6: Create a GitHub Repository
  - Task 7: Commit the Summary
  - Task 8: Create a Feature Branch & Add Metadata
  - Task 9: Open a Pull Request
  - Task 10: Commit the X API Script
  - Task 11: Update the README
  - Activity #1: Multi-Account Comparison Pipeline

---

# 🤝 Breakout Room 
## Setup, Tools & Agent Construction

## Task 1: Dependencies & Environment

We need:
- `langchain-mcp-adapters` to connect to MCP servers and convert their tools into LangChain tools
- `langgraph` for our agent graph with memory
- `langchain-openai` for our LLM
- `requests` for the X API calls
- `nest-asyncio` for async MCP operations inside Jupyter

> NOTE: Create a `.env` file in this directory with `X_BEARER_TOKEN`, `OPENAI_API_KEY`, and `GITHUB_PAT` before running.
> 
> Setup references:
> - GitHub fine-grained PAT guide: [Creating a personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens#creating-a-fine-grained-personal-access-token)
> - X API Bearer Token setup: [X Developer Portal](https://developer.x.com/en/portal/dashboard) and [X API access tiers](https://developer.x.com/en/products/twitter-api)


In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("X_BEARER_TOKEN"):
    os.environ["X_BEARER_TOKEN"] = getpass.getpass("Enter your X Bearer Token:")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

if not os.environ.get("GITHUB_PAT"):
    os.environ["GITHUB_PAT"] = getpass.getpass("Enter your GitHub PAT:")

In [2]:
import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

### Getting Your Credentials

**GitHub PAT (fine-grained):**
1. Open [GitHub Personal Access Tokens (fine-grained)](https://github.com/settings/personal-access-tokens/new).
2. Follow [GitHub's PAT setup guide](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens#creating-a-fine-grained-personal-access-token).
3. Set repository permissions to at least:
   - `Contents`: Read and write
   - `Pull requests`: Read and write
   - `Metadata`: Read-only

**X Bearer Token:**
1. Open the [X Developer Portal](https://developer.x.com/en/portal/dashboard).
2. Create/select a Project + App, then go to **Keys and Tokens** to generate a Bearer Token.
3. Confirm your plan supports the recent search endpoint (`GET /2/tweets/search/recent`) from the [X API product page](https://developer.x.com/en/products/twitter-api).


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# Test the connection
response = llm.invoke("Say 'MCP agent ready!' in exactly those words.")
print(response.content)

MCP agent ready!


## Task 2: X API as LangChain Tools

Instead of relying on a community-built MCP server for X, we'll call the **X API v2** directly and wrap our functions with the `@tool` decorator. This makes them available to our LangGraph agent as callable tools — just like the MCP tools will be.

This is a key architectural decision: **not everything needs to be an MCP server**. Wrapping a simple API call as a `@tool` is often simpler and more transparent.

**📚 Documentation:**
- [LangChain Tools Conceptual Guide](https://python.langchain.com/docs/concepts/tools/)
- [X API v2 Documentation](https://developer.x.com/en/docs/x-api)

In [20]:
import requests
import json
from langchain_core.tools import tool

BEARER_TOKEN = os.environ.get("X_BEARER_TOKEN")


@tool
def search_recent_posts(query: str, max_results: int = 20) -> str:
    """Search recent X/Twitter posts using the v2 API.
    Returns posts from the last 7 days matching the query.
    Use this for keyword searches, hashtag searches, or general topic searches.

    Args:
        query: The search query (e.g., 'AI safety', '#machinelearning', 'from:AndrewYNg')
        max_results: Number of results to return (10-100, default 20)
    """
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {BEARER_TOKEN}"}
    params = {
        "query": query,
        "max_results": min(max(max_results, 10), 100),
        "tweet.fields": "created_at,public_metrics,author_id,text",
        "expansions": "author_id",
        "user.fields": "name,username",
    }

    response = requests.get(url, headers=headers, params=params, verify=False)
    response.raise_for_status()
    data = response.json()

    tweets = data.get("data", [])
    if not tweets:
        return "No posts found for this query."

    result_lines = [f"Found {len(tweets)} posts:\n"]
    for t in tweets:
        metrics = t.get("public_metrics", {})
        result_lines.append(
            f"[{t.get('created_at', 'unknown')[:10]}] "
            f"{t['text'][:200]}\n"
            f"  Likes: {metrics.get('like_count', 0)} | "
            f"Retweets: {metrics.get('retweet_count', 0)}"
        )
    return "\n\n".join(result_lines)


@tool
def get_user_posts(username: str, max_results: int = 20) -> str:
    """Get recent original posts (no retweets) from a specific X/Twitter user.
    Use this when you want to see what a specific account has been posting.

    Args:
        username: The X/Twitter handle without the @ sign (e.g., 'AndrewYNg')
        max_results: Number of results to return (10-100, default 20)
    """
    query = f"from:{username} -is:retweet"
    return search_recent_posts.invoke({"query": query, "max_results": max_results})


x_api_tools = [search_recent_posts, get_user_posts]
print(f"Created {len(x_api_tools)} X API tools: {[t.name for t in x_api_tools]}")

Created 2 X API tools: ['search_recent_posts', 'get_user_posts']


In [21]:
import re
from langchain_core.tools import tool

MOCK_POSTS = [
    {
        "id": "1",
        "created_at": "2026-03-05T10:15:00Z",
        "author_username": "llm_wizard",
        "author_name": "LLM Wizard",
        "text": "RAG works better when you combine semantic retrieval with lexical retrieval and reranking.",
        "public_metrics": {"like_count": 42, "retweet_count": 8},
    },
    {
        "id": "2",
        "created_at": "2026-03-05T14:40:00Z",
        "author_username": "AndrewYNg",
        "author_name": "Andrew Ng",
        "text": "AI education remains one of the most important ways to broaden access to opportunity.",
        "public_metrics": {"like_count": 120, "retweet_count": 30},
    },
    {
        "id": "3",
        "created_at": "2026-03-04T09:20:00Z",
        "author_username": "llm_wizard",
        "author_name": "LLM Wizard",
        "text": "ParentDocumentRetriever can improve context quality, but chunking choices matter a lot.",
        "public_metrics": {"like_count": 35, "retweet_count": 6},
    },
    {
        "id": "4",
        "created_at": "2026-03-03T18:05:00Z",
        "author_username": "machinelearner",
        "author_name": "ML Research Notes",
        "text": "Hybrid search often outperforms pure dense retrieval on enterprise document search benchmarks.",
        "public_metrics": {"like_count": 87, "retweet_count": 19},
    },
    {
        "id": "5",
        "created_at": "2026-03-02T07:55:00Z",
        "author_username": "faithandlife",
        "author_name": "Faith & Life",
        "text": "Questions of life and faith are often explored through community, scripture, and personal reflection.",
        "public_metrics": {"like_count": 18, "retweet_count": 3},
    },
    {
        "id": "6",
        "created_at": "2026-03-01T12:10:00Z",
        "author_username": "llm_wizard",
        "author_name": "LLM Wizard",
        "text": "If your retriever recall is high but precision is low, add reranking before generation.",
        "public_metrics": {"like_count": 51, "retweet_count": 11},
    },
]


def _matches_query(post: dict, query: str) -> bool:
    q = query.strip()
    text = post["text"].lower()
    username = post["author_username"].lower()

    # Support a simple subset of X query syntax
    from_match = re.search(r"from:([A-Za-z0-9_]+)", q, re.IGNORECASE)
    if from_match:
        expected_user = from_match.group(1).lower()
        if username != expected_user:
            return False
        q = re.sub(r"from:[A-Za-z0-9_]+", "", q, flags=re.IGNORECASE).strip()

    q = re.sub(r"-is:retweet", "", q, flags=re.IGNORECASE).strip()

    if not q:
        return True

    # Split remaining query into simple terms and require all to match
    terms = [t.lower() for t in q.split() if t.strip()]
    return all(term in text or term in username for term in terms)


def _format_posts(posts: list[dict]) -> str:
    if not posts:
        return "No posts found for this query."

    result_lines = [f"Found {len(posts)} posts:\n"]
    for t in posts:
        metrics = t.get("public_metrics", {})
        result_lines.append(
            f"[{t.get('created_at', 'unknown')[:10]}] "
            f"@{t.get('author_username', 'unknown')}: "
            f"{t['text'][:200]}\n"
            f"  Likes: {metrics.get('like_count', 0)} | "
            f"Retweets: {metrics.get('retweet_count', 0)}"
        )
    return "\n\n".join(result_lines)


@tool
def search_recent_posts(query: str, max_results: int = 20) -> str:
    """Search recent mock X/Twitter posts.
    Returns posts from a local mock dataset matching the query.

    Args:
        query: The search query (e.g., 'AI safety', '#machinelearning', 'from:AndrewYNg')
        max_results: Number of results to return (default 20)
    """
    limit = min(max(max_results, 1), 100)
    matched = [post for post in MOCK_POSTS if _matches_query(post, query)]
    matched = sorted(matched, key=lambda x: x["created_at"], reverse=True)[:limit]
    return _format_posts(matched)


@tool
def get_user_posts(username: str, max_results: int = 20) -> str:
    """Get recent original mock posts from a specific X/Twitter user."""
    query = f"from:{username} -is:retweet"
    return search_recent_posts.invoke({"query": query, "max_results": max_results})


x_api_tools = [search_recent_posts, get_user_posts]
print(f"Created {len(x_api_tools)} X API tools: {[t.name for t in x_api_tools]}")

Created 2 X API tools: ['search_recent_posts', 'get_user_posts']


Let's verify our X API tools work before wiring them into the agent:

In [22]:
# Quick test — fetch recent posts from a public account
result = get_user_posts.invoke({"username": "llm_wizard", "max_results": 10})
print(result[:500])

Found 3 posts:


[2026-03-05] @llm_wizard: RAG works better when you combine semantic retrieval with lexical retrieval and reranking.
  Likes: 42 | Retweets: 8

[2026-03-04] @llm_wizard: ParentDocumentRetriever can improve context quality, but chunking choices matter a lot.
  Likes: 35 | Retweets: 6

[2026-03-01] @llm_wizard: If your retriever recall is high but precision is low, add reranking before generation.
  Likes: 51 | Retweets: 11


## Task 3: Connect to GitHub MCP Server & Load Tools

Now we'll connect to the **GitHub MCP Server** — an official, GitHub-maintained MCP server that gives agents the ability to manage repositories, issues, pull requests, and more.

We use `langchain-mcp-adapters` to:
1. Connect to the remote GitHub MCP server over HTTP
2. Automatically convert all MCP tools into LangChain-compatible tools

This is the key MCP integration point — instead of writing custom GitHub API wrappers, we get a full set of tools for free just by connecting to the MCP server.

**📚 Documentation:**
- [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)
- [GitHub MCP Server](https://github.com/github/github-mcp-server)
- [Model Context Protocol Specification](https://modelcontextprotocol.io/)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect to the GitHub MCP server using Streamable HTTP transport
# The server exposes GitHub operations as MCP tools that our agent can call
mcp_client = MultiServerMCPClient(
    {
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {os.environ['GITHUB_PAT']}",
            },
            
        }
    }
)

# Load all tools from the MCP server
github_mcp_tools = await mcp_client.get_tools()

print(f"Loaded {len(github_mcp_tools)} GitHub MCP tools:\n")
for t in github_mcp_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Loaded 40 GitHub MCP tools:

  - add_comment_to_pending_review: Add review comment to the requester's latest pending pull request review. A pend...
  - add_issue_comment: Add a comment to a specific issue in a GitHub repository. Use this tool to add c...
  - add_reply_to_pull_request_comment: Add a reply to an existing pull request comment. This creates a new comment that...
  - create_branch: Create a new branch in a GitHub repository...
  - create_or_update_file: Create or update a single file in a GitHub repository. 
If updating, you should ...
  - create_pull_request: Create a new pull request in a GitHub repository....
  - create_repository: Create a new GitHub repository in your account or specified organization...
  - delete_file: Delete a file from a GitHub repository...
  - fork_repository: Fork a GitHub repository to your account or specified organization...
  - get_commit: Get details for a commit from a GitHub repository...
  - get_file_contents: Get the contents of a file 

### Key GitHub MCP Tools

The MCP server exposes many tools, but the key ones we'll use are:

| MCP Tool | Replaces (Git CLI) | What It Does |
|---|---|---|
| `create_repository` | `git init` + GitHub UI | Creates a new repo on your account |
| `create_or_update_file` | `git add` + `git commit` + `git push` | Commits a file directly to a branch |
| `create_branch` | `git checkout -b` | Creates a new branch |
| `create_pull_request` | `gh pr create` | Opens a PR from one branch to another |
| `search_repositories` | `gh repo list` | Searches across your repos |
| `get_file_contents` | `git show` / `cat` | Reads a file from a repo |
| `list_commits` | `git log` | Shows commit history |

## Task 4: Build the LangGraph Agent with Memory

Now we combine **both tool sets** into a single LangGraph agent:
- **X API tools** — custom `@tool` functions for searching posts
- **GitHub MCP tools** — loaded from the MCP server via `langchain-mcp-adapters`

We add **`MemorySaver`** for short-term memory so the agent remembers context across the multi-step workflow (e.g., it fetches posts in one turn, summarizes them in the next, and commits the summary in a third).

The architecture follows the standard LangGraph ReAct pattern from Sessions 4-6:

```
┌─────────┐     ┌───────────┐
│  START   │────▶│   Agent   │◀──────────────┐
└─────────┘     │  (LLM +   │               │
                │   tools)  │               │
                └─────┬─────┘               │
                      │                     │
               has tool calls?              │
                /           \               │
              yes            no             │
              /               \             │
    ┌─────────────┐     ┌─────────┐        │
    │  Tool Node  │     │   END   │        │
    │ (X API +    │     └─────────┘        │
    │  GitHub MCP)│─────────────────────────┘
    └─────────────┘
```

**📚 Documentation:**
- [LangGraph ReAct Agent](https://langchain-ai.github.io/langgraph/tutorials/introduction/)
- [MemorySaver (Checkpointing)](https://langchain-ai.github.io/langgraph/concepts/persistence/)
- [ToolNode Prebuilt](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.ToolNode)

In [28]:
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Combine all tools: X API tools + GitHub MCP tools
all_tools = x_api_tools + github_mcp_tools
print(f"Total tools available to agent: {len(all_tools)}")
print(f"  X API tools: {[t.name for t in x_api_tools]}")
print(f"  GitHub MCP tools: {[t.name for t in github_mcp_tools]}")

Total tools available to agent: 42
  X API tools: ['search_recent_posts', 'get_user_posts']
  GitHub MCP tools: ['add_comment_to_pending_review', 'add_issue_comment', 'add_reply_to_pull_request_comment', 'create_branch', 'create_or_update_file', 'create_pull_request', 'create_repository', 'delete_file', 'fork_repository', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_me', 'get_release_by_tag', 'get_tag', 'get_team_members', 'get_teams', 'issue_read', 'issue_write', 'list_branches', 'list_commits', 'list_issue_types', 'list_issues', 'list_pull_requests', 'list_releases', 'list_tags', 'merge_pull_request', 'pull_request_read', 'pull_request_review_write', 'push_files', 'request_copilot_review', 'search_code', 'search_issues', 'search_pull_requests', 'search_repositories', 'search_users', 'sub_issue_write', 'update_pull_request', 'update_pull_request_branch']


In [56]:
# Step 1: Define the Agent State
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# Step 2: Define the system prompt
SYSTEM_PROMPT = """You are an AI assistant that can search X/Twitter posts and manage GitHub repositories.

You have two categories of tools:
1. X API tools: search_recent_posts, get_user_posts — for searching and retrieving X/Twitter posts
2. GitHub MCP tools: for creating repos, committing files, creating branches, opening PRs, etc.

When asked to summarize posts, retrieve them first using the X API tools, then provide a structured
markdown summary with: Overview, Key Themes, Notable Posts, and Summary Statistics.

When asked to perform GitHub operations, use the appropriate GitHub MCP tool.
Always use the available tools when appropriate. Be concise in your responses."""


# Step 3: Bind tools to the LLM
llm_with_tools = llm.bind_tools(all_tools)


# Step 4: Define the agent node
def agent_node(state: AgentState):
    """The agent node — calls the LLM with the current conversation and available tools."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


# Step 5: Define the tool node
tool_node = ToolNode(all_tools, handle_tool_errors=True)


# Step 6: Define routing logic
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """Determine whether to call tools or end the conversation."""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "end"


# Step 7: Build the graph
workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
workflow.add_edge("tools", "agent")

# Compile with MemorySaver for short-term memory across turns
checkpointer = MemorySaver()
agent = workflow.compile(checkpointer=checkpointer)

print("Agent compiled with memory and tools!")

Agent compiled with memory and tools!


Let's visualize the graph to confirm our architecture:

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

### Helper function for running the agent

We'll use a single `thread_id` throughout the notebook so the agent remembers previous interactions (short-term memory via the checkpointer).

In [57]:
# Use a consistent thread_id so the agent remembers context across all tasks
config = {"configurable": {"thread_id": "mcp-workflow-1"}}


async def ask_agent(user_message: str) -> str:
    """Send a message to the agent and return its final response."""
    response = await agent.ainvoke(
        {"messages": [HumanMessage(content=user_message)]},
        config,
    )
    return response["messages"][-1].content

## Task 5: Test the Agent — Search & Summarize X Posts

Let's put the agent through its paces. First, we'll ask it to search for posts and generate a summary. Because we're using `MemorySaver` with a consistent `thread_id`, the agent will remember the posts it found when we ask it to summarize them.

In [50]:
# Ask the agent to fetch posts — it will use the get_user_posts tool
result = await ask_agent("Get recent posts from @llm_wizard on X/Twitter.")
print(result[:1000])

I retrieved 3 recent posts from @llm_wizard on X/Twitter. Would you like me to generate a summary of these posts for the summary.md file in your GitHub repo?


In [51]:
# Ask the agent to summarize — it remembers the posts from the previous turn!
summary = await ask_agent(
    "Now summarize those posts into a structured markdown report with sections for: "
    "Overview, Key Themes, Notable Posts, and Summary Statistics. "
    "Format it so it can be saved directly as a summary.md file."
)
print(summary)

Here is the structured markdown summary report for the recent posts from @llm_wizard:

```markdown
# Summary of @llm_wizard's Recent 2026 X Posts

## Overview
This report summarizes key insights from the recent posts of @llm_wizard, a thought leader in retrieval-augmented generation (RAG) and document retrieval techniques. The posts focus on improving retrieval quality and generation precision through various methods.

## Key Themes
- Combining semantic and lexical retrieval methods enhances RAG performance.
- The choice of chunking strategy significantly impacts context quality in document retrieval.
- Adding reranking steps can improve precision when retriever recall is high but precision is low.

## Notable Posts
- "RAG works better when you combine semantic retrieval with lexical retrieval and reranking." (2026-03-05)
- "ParentDocumentRetriever can improve context quality, but chunking choices matter a lot." (2026-03-04)
- "If your retriever recall is high but precision is low, add

Save the summary locally for reference:

In [52]:
with open("summary.md", "w") as f:
    f.write(summary)

print("Summary saved to summary.md")

Summary saved to summary.md


### ❓ Question #1:



##### Answer:

*Your answer here*

### 🏗️ Activity #1:

Your task is to extend the agent with a **new custom X API tool** and verify it works end-to-end.

1. **Create a new `@tool` function** called `get_user_profile` that retrieves a user's public profile information using the X API v2 [`GET /2/users/by/username/:username`](https://developer.x.com/en/docs/x-api/users/lookup/api-reference/get-users-by-username-username) endpoint. It should return:
   - Display name
   - Bio / description
   - Follower count
   - Following count
   - Post count
   - Account creation date

2. **Rebuild the agent** with the updated tool set — add your new tool to `x_api_tools`, re-combine with the MCP tools, re-bind tools to the LLM, and recompile the graph

3. **Test it** by asking the agent to:
   - Retrieve the profile of an AI thought leader of your choice
   - Compare that profile with the posts you already retrieved in Task 5 — does the bio match the posting themes?

> HINT: The X API v2 user lookup endpoint uses the same Bearer Token authentication. You'll need `user.fields=description,public_metrics,created_at` in your request params.

In [37]:
from langchain_core.tools import tool

MOCK_USER_PROFILES = {
    "andrewyng": {
        "display_name": "Andrew Ng",
        "description": "Founder of DeepLearning.AI, Landing AI, and AI Fund. Working to make AI accessible and useful for everyone.",
        "followers_count": 1250000,
        "following_count": 980,
        "tweet_count": 18500,
        "created_at": "2008-06-18T00:00:00Z",
    },
    "llm_wizard": {
        "display_name": "LLM Wizard",
        "description": "Sharing practical insights on RAG, retrievers, reranking, and production LLM systems.",
        "followers_count": 12400,
        "following_count": 210,
        "tweet_count": 1340,
        "created_at": "2022-11-03T00:00:00Z",
    },
    "machinelearner": {
        "display_name": "ML Research Notes",
        "description": "Notes on machine learning, evaluation, retrieval systems, and applied AI.",
        "followers_count": 48200,
        "following_count": 320,
        "tweet_count": 6120,
        "created_at": "2019-02-14T00:00:00Z",
    },
    "faithandlife": {
        "display_name": "Faith & Life",
        "description": "Exploring meaning, community, scripture, and reflection in everyday life.",
        "followers_count": 8700,
        "following_count": 145,
        "tweet_count": 920,
        "created_at": "2021-04-09T00:00:00Z",
    },
}


@tool
def get_user_profile(username: str) -> str:
    """Get a user's public profile information from a mock X/Twitter dataset.

    Args:
        username: The X/Twitter handle without the @ sign.
    """
    key = username.lower().lstrip("@")
    profile = MOCK_USER_PROFILES.get(key)

    if not profile:
        return f"No profile found for @{username}."

    return (
        f"Profile for @{key}\n"
        f"Display name: {profile['display_name']}\n"
        f"Bio: {profile['description']}\n"
        f"Followers: {profile['followers_count']}\n"
        f"Following: {profile['following_count']}\n"
        f"Post count: {profile['tweet_count']}\n"
        f"Created at: {profile['created_at']}"
    )

In [41]:
from langchain.agents import create_agent

# Step 1: Define the Agent State
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# Step 2: Define the system prompt
SYSTEM_PROMPT = """You are an AI assistant that can search X/Twitter posts and manage GitHub repositories.

You have two categories of tools:
1. X API tools: search_recent_posts, get_user_posts — for searching and retrieving X/Twitter posts
2. GitHub MCP tools: for creating repos, committing files, creating branches, opening PRs, etc.

When asked to summarize posts, retrieve them first using the X API tools, then provide a structured
markdown summary with: Overview, Key Themes, Notable Posts, and Summary Statistics.

When asked to perform GitHub operations, use the appropriate GitHub MCP tool.
Always use the available tools when appropriate. Be concise in your responses."""

x_api_tools.append(get_user_profile)
# Step 3: Bind tools to the LLM
all_tools = x_api_tools + github_mcp_tools
llm_with_tools = llm.bind_tools(all_tools)


agent = create_agent(
    model=llm_with_tools,
    tools=all_tools,
    checkpointer=checkpointer,
    system_prompt=SYSTEM_PROMPT,
)

In [43]:
response = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Get the profile of AndrewYNg, then get his recent posts, "
                    "and tell me whether his bio matches the posting themes."
                ),
            }
        ]
    },
    config={"configurable": {"thread_id": "test-x-profile-1"}},
)

print(response["messages"][-1].content)

Andrew Ng's profile bio states: "Founder of DeepLearning.AI, Landing AI, and AI Fund. Working to make AI accessible and useful for everyone."

His recent post is about AI education being important to broaden access to opportunity.

The bio matches the posting theme well, as both emphasize AI and its accessibility and usefulness for people. The post aligns with his mission of making AI accessible and impactful.


---

## Phase 2: MCP Workflow Through the Agent

Now we'll use the same agent to perform all GitHub repository operations through the **GitHub MCP tools**. Because the agent has memory, it already knows the summary it generated in Phase 1.

Each task below sends a natural language instruction to the agent. The agent decides which GitHub MCP tool(s) to call to fulfill the request.

## Task 6: Create a New Repository

The agent will use the `create_repository` MCP tool to create a new repo on your GitHub account.

In [47]:
result = await ask_agent(
    "Using your GitHub tools, create a new public repository on my account called "
    "`x-post-summarizer-2026`. Add a description: 'AI-generated summary of a public "
    "figure's 2026 X posts, built with LangGraph, MCP tools, and the X API.' "
    "Initialize it with a README."
)
print(result)

I have created the public repository "x-post-summarizer-2026" on your GitHub account with the description: "AI-generated summary of a public figure's 2026 X posts, built with LangGraph, MCP tools, and the X API." It is initialized with a README file. You can view it here: https://github.com/dznidaric/x-post-summarizer-2026.


## Task 7: Commit the Summary to Your Repo

The agent remembers the summary it generated earlier (short-term memory) and can commit it directly.

In [60]:
result = await ask_agent(
    "Using your GitHub tools, create a new file called `summary.md` in the "
    "`x-post-summarizer-2026` repo on the `main` branch. The file should contain "
    "the X post summary you generated earlier. Use the commit message: "
    "'Add 2026 X post summary'."
    """summary.md: Here is the structured markdown summary report for the recent posts from @llm_wizard:

```markdown
# Summary of @llm_wizard's Recent 2026 X Posts

## Overview
This report summarizes key insights from the recent posts of @llm_wizard, a thought leader in retrieval-augmented generation (RAG) and document retrieval techniques. The posts focus on improving retrieval quality and generation precision through various methods.

## Key Themes
- Combining semantic and lexical retrieval methods enhances RAG performance.
- The choice of chunking strategy significantly impacts context quality in document retrieval.
- Adding reranking steps can improve precision when retriever recall is high but precision is low.

## Notable Posts
- "RAG works better when you combine semantic retrieval with lexical retrieval and reranking." (2026-03-05)
- "ParentDocumentRetriever can improve context quality, but chunking choices matter a lot." (2026-03-04)
- "If your retriever recall is high but precision is low, add reranking before generation." (2026-03-01)

## Summary Statistics
- Total posts analyzed: 3
- Total likes received: 128
- Total retweets received: 25
```

I can now create the summary.md file in your GitHub repo with this content if you want."""
)
print(result)

It seems the repository "x-post-summarizer-2026" under the user "user" does not exist or I do not have access to it. Could you please confirm the correct repository owner and name or provide access details?


## Task 8: Create a Feature Branch and Add Metadata

The agent will use `create_branch` and `create_or_update_file` MCP tools.

In [ ]:
result = await ask_agent(
    "Create a new branch called `add-metadata` in my `x-post-summarizer-2026` repo. "
    "On that branch, create a file called `metadata.json` that contains: the account "
    "handle analyzed, the date range of posts, the number of posts analyzed, and the "
    "top 5 themes identified from the summary. Commit it with the message "
    "'Add analysis metadata'."
)
print(result)

## Task 9: Open a Pull Request

The agent will use the `create_pull_request` MCP tool.

In [ ]:
result = await ask_agent(
    "Open a pull request in my `x-post-summarizer-2026` repo from the `add-metadata` "
    "branch to `main`. Title it 'Add analysis metadata' and include a description "
    "summarizing what the metadata file contains."
)
print(result)

## Task 10: Commit the X API Script

We'll ask the agent to commit a clean version of the X search script — reading credentials from environment variables, no hardcoded keys.

In [ ]:
x_search_script = '''import requests
import json
import os
from datetime import datetime

BEARER_TOKEN = os.environ.get("X_BEARER_TOKEN")

def search_recent_posts(query: str, max_results: int = 20) -> dict:
    """Search recent X posts using the v2 API."""
    url = "https://api.x.com/2/tweets/search/recent"
    headers = {"Authorization": f"Bearer {BEARER_TOKEN}"}
    params = {
        "query": query,
        "max_results": min(max_results, 100),
        "tweet.fields": "created_at,public_metrics,author_id,text",
        "expansions": "author_id",
        "user.fields": "name,username",
    }
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()

def get_user_posts(username: str, max_results: int = 20) -> dict:
    """Get recent posts from a specific user."""
    query = f"from:{username} -is:retweet"
    return search_recent_posts(query, max_results)

if __name__ == "__main__":
    import sys
    handle = sys.argv[1] if len(sys.argv) > 1 else "llM_wizard"
    print(f"Searching for recent posts from @{handle}...")
    results = get_user_posts(handle)
    with open("posts.json", "w") as f:
        json.dump(results, f, indent=2)
    tweets = results.get("data", [])
    print(f"Found {len(tweets)} posts.")
    for tweet in tweets:
        print(f"  [{tweet[\"created_at\"][:10]}] {tweet[\"text\"][:100]}...")
'''

result = await ask_agent(
    f"Using your GitHub tools, create a new file called `x_search.py` in the "
    f"`x-post-summarizer-2026` repo on the `main` branch. Use the commit message: "
    f"'Add X API search script'. Here is the file content:\n\n{x_search_script}"
)
print(result)

## Task 11: Update the README

The agent already knows everything about the project from its conversation memory — what account was analyzed, how the project was built, etc.

In [ ]:
result = await ask_agent(
    "Update the README.md in my `x-post-summarizer-2026` repo on main to include: "
    "a project description explaining this repo summarizes a public figure's 2026 X "
    "posts using AI, the handle analyzed, how the project was built (using a LangGraph "
    "agent with GitHub MCP tools for repo operations and the X API v2 for post "
    "retrieval), and instructions for someone else to replicate the process — including "
    "how to set up their X API Bearer Token and install Python dependencies."
)
print(result)

### ❓ Question #2:

Compare using GitHub MCP tools (through a LangGraph agent) to traditional `git` commands. What felt easier? What felt harder or less transparent?

##### Answer:
Using GitHub MCP tools through a LangGraph agent felt easier in terms of automation and natural language interaction. Instead of remembering exact git commands, I could simply describe the task (for example, “create a file in the repository and commit it”), and the agent handled selecting the appropriate tool and executing the steps. This makes repository operations more accessible and integrates well with other AI-driven workflows, such as generating summaries or content before committing them.

However, using MCP tools also felt less transparent and sometimes harder to debug compared to traditional git. With git, every step—clone, add, commit, push—is explicit and visible, so it’s easier to understand what is happening under the hood. With MCP tools, many of those steps are abstracted away, which is convenient but can make it harder to diagnose issues such as permission errors, failed API calls, or missing tool responses. Additionally, the agent workflow introduces extra complexity around tool calls, message sequencing, and state management that does not exist when using simple git commands.

The worst thing is when you get error on one call the agent remembers that error and even after fixing the PAT permissions, the next model call still sees an earlier assistant message with an unresolved tool_call_id, and OpenAI rejects it.

### ❓ Question #3:

You used MCP for GitHub but wrapped the X API as a `@tool` directly. What are the tradeoffs of consuming an API through an MCP server versus wrapping it as a LangChain tool? When would each approach make more sense?

##### Answer:
Using an MCP server provides a standardized and reusable way to expose many API capabilities to agents. It separates the tool logic from the agent code, which makes it easier to reuse across projects and manage skills we want centrally. However, it adds more infrastructure and can be harder to debug.

Wrapping an API directly as a LangChain @tool is simpler and faster to implement. It keeps the logic in your codebase and is easier to understand and debug, but it can become harder to maintain if the API grows or needs to be reused across multiple systems.

In general, MCP is better for large, reusable integrations, while direct tools are better for small, application-specific APIs.

### 🏗️ Activity #1:

Your task is to extend the MCP workflow by building a **Multi-Account Comparison Pipeline** through the agent.

You are expected to:

1. **Retrieve posts from a second X account** — choose another public figure or thought leader in a related field

2. **Generate a structured comparison** by asking the agent to create a `comparison.md` file that includes:
   - Side-by-side topic analysis for both accounts
   - Tone and sentiment differences
   - Posting frequency comparison
   - Top 3 most notable posts from each account
   - A brief conclusion about each account's focus area

3. **Commit through the MCP workflow**:
   - Create a new branch called `add-comparison` in your `x-post-summarizer-2026` repo
   - Commit `comparison.md` to that branch
   - Open a pull request to merge it into `main`

> NOTE: The agent already has memory of the first account's posts from Phase 1. You only need to fetch posts from the second account — the agent will use its memory for the rest.

In [61]:
result = await ask_agent(
    "Get recent original posts from @machinelearner."
)
print(result)

Recent original post from @machinelearner:

- Date: 2026-03-03
- Content: "Hybrid search often outperforms pure dense retrieval on enterprise document search benchmarks."
- Likes: 87
- Retweets: 19

If you want more details or posts, let me know!


In [64]:
result = await ask_agent(
    "First account: Andrew Ng's profile bio states: 'Founder of DeepLearning.AI, Landing AI, and AI Fund. Working to make AI accessible and useful for everyone'. His recent post is about AI education being important to broaden access to opportunity.The bio matches the posting theme well, as both emphasize AI and its accessibility and usefulness for people. The post aligns with his mission of making AI accessible and impactful."
    "Using your memory of the first account and the recent posts from @machinelearner, "
    "create a structured comparison in a file called `comparison.md` with:\n"
    "- side-by-side topic analysis for both accounts\n"
    "- tone and sentiment differences\n"
    "- posting frequency comparison\n"
    "- top 3 most notable posts from each account\n"
    "- a brief conclusion about each account's focus area\n\n"
    "GitHub user: dznidaric\n"
    "Then use your GitHub tools to:\n"
    "1. create a new branch called `add-comparison` in the `x-post-summarizer-2026` repo\n"
    "2. add `comparison.md` to that branch\n"
    "3. commit it\n"
    "4. open a pull request into `main`."
)
print(result)

The branch "add-comparison" already exists in the repository. I will proceed to add the file comparison.md to this existing branch, commit it, and then open a pull request into main. First, I will create the content for comparison.md based on the data retrieved.
Here is the content for comparison.md based on the profiles and recent posts of Andrew Ng and @machinelearner:

# Comparison of Andrew Ng and @machinelearner on X/Twitter

## Side-by-Side Topic Analysis
| Aspect               | Andrew Ng                                   | @machinelearner                          |
|----------------------|---------------------------------------------|-----------------------------------------|
| Bio Focus            | AI accessibility, AI education, AI impact  | Machine learning research, evaluation, retrieval systems, applied AI |
| Recent Post Topic    | Importance of AI education for opportunity | Hybrid search outperforming dense retrieval in enterprise document search |
| Emphasis          